<a href="https://colab.research.google.com/github/KijoSal-dev/Natural_language_processing_with_transformers/blob/main/Salome_Kungu_CS_DA03_26054_NLP_with_transformers_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# STEP 1: Install compatible transformers (TF support is stable in 4.x)
!pip install -q "transformers==4.48.3"
!pip install -q "tokenizers>=0.22.0,<=0.23.0"
!pip install -q --upgrade "scikit-learn"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 44.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 33.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.48.3 requires tokenizers<0.22,>=0.21, but you have tokenizers 0.22.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# STEP 2: Import libraries
# Re-install tokenizers to satisfy transformers requirement
!pip uninstall -y tokenizers
!pip install -q "tokenizers>=0.21.0,<0.22.0"

from transformers import BertTokenizer, TFBertModel
import tensorflow as tf
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries loaded successfully!")
print("TensorFlow version:", tf.__version__)
print("Transformers version:", "transformers" in __import__("__main__").__dict__ or "ok")

Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Libraries loaded successfully!
TensorFlow version: 2.20.0
Transformers version: ok


In [3]:
# STEP 3: Load Pretrained BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = TFBertModel.from_pretrained('bert-base-uncased')

print("BERT loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

BERT loaded successfully!


In [4]:
# STEP 4: Define Sentence Pairs

sentence_pairs = [

    # Provided pairs
    ("How do I learn Python?", "What is the best way to study Python?"),
    ("What is AI?", "How to cook pasta?"),
    ("How do I bake a chocolate cake?", "Give me a chocolate cake recipe."),
    ("How can I improve my coding skills?", "Tips for becoming better at programming."),
    ("Where can I buy cheap laptops?", "Best sites to find affordable computers."),

    # Added pairs
    ("What is machine learning?", "Explain machine learning concepts."),
    ("How do I reset my password?", "Steps to change my account password."),
    ("What causes rain?", "Why does precipitation occur?"),
    ("Best exercises for weight loss", "How to install Windows 10"),
    ("Tips for studying data science", "Advice for learning data science")
]

In [5]:
# STEP 5: Define Ground Truth Labels: 1 = similar, 0 = not similar

labels = [1,0,1,1,1, 1,1,1,0,1]

In [6]:
# STEP 6 :Function to get the BERT [CLS] embedding for a sentence

def get_sentence_embedding(sentence):
    # Tokenize and encode sentence into input tensors
    inputs = tokenizer(sentence, return_tensors='tf', padding=True, truncation=True)
    # Get model output
    outputs = bert_model(inputs)
    # Extract [CLS] token embedding (shape: [1, 768])
    cls_embedding = outputs.last_hidden_state[:, 0, :]
    return cls_embedding.numpy()

In [7]:
# STEP 7: Calculate cosine similarity for each pair
predictions = []
for sent1, sent2 in sentence_pairs:
    emb1 = get_sentence_embedding(sent1)
    emb2 = get_sentence_embedding(sent2)
    sim_score = cosine_similarity(emb1, emb2)[0][0]
    pred = 1 if sim_score > 0.7 else 0
    predictions.append(pred)

    print(f"\nSentence 1: {sent1}")
    print(f"Sentence 2: {sent2}")
    print(f"Cosine Similarity: {sim_score:.4f} → Predicted Similar: {pred}")


Sentence 1: How do I learn Python?
Sentence 2: What is the best way to study Python?
Cosine Similarity: 0.9743 → Predicted Similar: 1

Sentence 1: What is AI?
Sentence 2: How to cook pasta?
Cosine Similarity: 0.9033 → Predicted Similar: 1

Sentence 1: How do I bake a chocolate cake?
Sentence 2: Give me a chocolate cake recipe.
Cosine Similarity: 0.8938 → Predicted Similar: 1

Sentence 1: How can I improve my coding skills?
Sentence 2: Tips for becoming better at programming.
Cosine Similarity: 0.8633 → Predicted Similar: 1

Sentence 1: Where can I buy cheap laptops?
Sentence 2: Best sites to find affordable computers.
Cosine Similarity: 0.8750 → Predicted Similar: 1

Sentence 1: What is machine learning?
Sentence 2: Explain machine learning concepts.
Cosine Similarity: 0.8034 → Predicted Similar: 1

Sentence 1: How do I reset my password?
Sentence 2: Steps to change my account password.
Cosine Similarity: 0.8712 → Predicted Similar: 1

Sentence 1: What causes rain?
Sentence 2: Why doe

In [8]:
# STEP 8: Evaluate accuracy
correct = 0
for i in range(len(predictions)):
    if predictions[i] == labels[i]:
        correct += 1

In [9]:
# STEP 9: Final accuracy calculation
total = len(labels)
accuracy = correct / total
print(f"\nAccuracy: {accuracy:.2%}")


Accuracy: 80.00%


In [10]:
# STEP 10: Final results
print("Predictions:", predictions)
print("Ground Truth:", labels)
print("Accuracy:", round(accuracy*100,2), "%")

Predictions: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Ground Truth: [1, 0, 1, 1, 1, 1, 1, 1, 0, 1]
Accuracy: 80.0 %


In [11]:
import os

drive_path = '/content/drive/MyDrive/Colab Notebooks'
if os.path.exists(drive_path):
    print(f"Contents of {drive_path}:")
    for item in os.listdir(drive_path):
        print(item)
else:
    print(f"Directory not found: {drive_path}")

Directory not found: /content/drive/MyDrive/Colab Notebooks


In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/Salome_Kungu_CS-DA03-26054_NLP_with_transformers_Project.ipynb"

# Load the notebook
with open(notebook_path, 'r') as f:
    nb = json.load(f)

# Remove the widgets metadata that's causing issues
if 'widgets' in nb.get('metadata', {}):
    del nb['metadata']['widgets']

# Save the cleaned notebook
with open(notebook_path, 'w') as f:
    json.dump(nb, f, indent=1)

print("Notebook cleaned! Widgets metadata removed.")

Notebook cleaned! Widgets metadata removed.
